In [1]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer
import datasets
import torch

/home/kolla/anaconda3/envs/verbosius/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
df = datasets.load_dataset("rotten_tomatoes")

train_x = df["train"]["text"]
test_x = df["test"]["text"]
train_y = df["train"]["label"]
test_y = df["test"]["label"]

Found cached dataset rotten_tomatoes (/home/kolla/.cache/huggingface/datasets/rotten_tomatoes/default/1.0.0/40d411e45a6ce3484deed7cc15b82a53dad9a72aafd9f86f8f227134bec5ca46)
100%|██████████| 3/3 [00:00<00:00, 420.71it/s]


In [12]:
val_x = test_x[40:50]
val_y = test_y[40:50]
train_x = train_x[:100]
test_x = test_x[:40]
train_y = train_y[:100]
test_y = test_y[:40]

In [13]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilroberta-base", num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
train_x_tokenized = tokenizer(train_x, padding=True, truncation=True, return_tensors="pt")
test_x_tokenized = tokenizer(test_x, padding=True, truncation=True, return_tensors="pt")

train_y = torch.tensor(train_y)
test_y = torch.tensor(test_y)

train_data = [{"input_ids" : j, "attention_mask" : k, "labels" : l} for j, k, l in zip(train_x_tokenized["input_ids"], train_x_tokenized["attention_mask"], train_y)]
test_data = [{"input_ids" : j, "attention_mask" : k, "labels" : l} for j, k, l in zip(test_x_tokenized["input_ids"], test_x_tokenized["attention_mask"], test_y)]

#test_dataset = torch.utils.data.TensorDataset(test_x_tokenized["input_ids"], test_x_tokenized["attention_mask"], test_y)
#train_dataset = torch.utils.data.TensorDataset(train_x_tokenized["input_ids"], train_x_tokenized["attention_mask"], train_y)

In [15]:
training_args = TrainingArguments(
    output_dir='/home/kolla/data/dump/',          
    num_train_epochs=1,              
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=None,
) 

trainer.train()

/home/kolla/anaconda3/envs/verbosius/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
100%|██████████| 13/13 [00:33<00:00,  2.56s/it]

{'train_runtime': 33.2833, 'train_samples_per_second': 3.005, 'train_steps_per_second': 0.391, 'train_loss': 0.11420844151423527, 'epoch': 1.0}


TrainOutput(global_step=13, training_loss=0.11420844151423527, metrics={'train_runtime': 33.2833, 'train_samples_per_second': 3.005, 'train_steps_per_second': 0.391, 'train_loss': 0.11420844151423527, 'epoch': 1.0})

In [16]:
val_x

["you needn't be steeped in '50s sociology , pop culture or movie lore to appreciate the emotional depth of haynes' work . though haynes' style apes films from the period . . . its message is not rooted in that decade .",
 "waiting for godard can be fruitful : 'in praise of love' is the director's epitaph for himself .",
 'a gangster movie with the capacity to surprise .',
 'the film has a laundry list of minor shortcomings , but the numerous scenes of gory mayhem are worth the price of admission . . . if " gory mayhem " is your idea of a good time .',
 'if not a home run , then at least a solid base hit .',
 'goldmember is funny enough to justify the embarrassment of bringing a barf bag to the moviehouse .',
 '. . . a fairly disposable yet still entertaining b picture .',
 "it may not be particularly innovative , but the film's crisp , unaffected style and air of gentle longing make it unexpectedly rewarding .",
 "the film truly does rescue [the funk brothers] from motown's shadows . 

In [19]:
samp_0 = tokenizer(val_x[0])

samp_0

{'input_ids': [101, 2017, 2342, 2078, 1005, 1056, 2022, 9561, 2098, 1999, 1005, 2753, 2015, 11507, 1010, 3769, 3226, 2030, 3185, 19544, 2000, 9120, 1996, 6832, 5995, 1997, 21805, 1005, 2147, 1012, 2295, 21805, 1005, 2806, 27754, 3152, 2013, 1996, 2558, 1012, 1012, 1012, 2049, 4471, 2003, 2025, 15685, 1999, 2008, 5476, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [21]:
trainer.predict([samp_0])

100%|██████████| 1/1 [00:00<00:00, 902.19it/s]


PredictionOutput(predictions=array([[-2.9536233,  2.6575868]], dtype=float32), label_ids=None, metrics={'test_runtime': 1.0105, 'test_samples_per_second': 0.99, 'test_steps_per_second': 0.99})

In [72]:
len(None)

TypeError: object of type 'NoneType' has no len()

In [82]:
class Add:

    def __init__(self, a, b):
        self.a = a
        self.b = b

    def add(self):
        
        self.a += 1
        self.b += 1

        return self.a, self.b
    
b = Add(1, 1)

In [91]:
b.add()

(10, 10)